In [11]:
import os
import cv2
import pandas as pd
import numpy as np
from ultralytics import YOLO
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.models import Model
from sklearn.preprocessing import StandardScaler

In [12]:
# Paths and Parameters
video_folder = '../sdata'
metadata_path = "/Users/jeremiah/video-p-reaction-joketype-modular/Cursor_october/dataa/excel.xlsx"
frame_sample_rate = 5  # Extract every 5th frame
window_size = 10       # Window size for temporal aggregation (number of frames)

# Load Metadata
metadata = pd.read_excel(metadata_path)
video_ids = metadata['VideoID'].tolist()
labels = metadata['HowFunny'].tolist()

In [13]:
# Load YOLOv8 Pose Estimation Model
try:
    pose_model = YOLO("yolov8n-pose")  # Adjust model as needed
except Exception as e:
    print(f"Error loading YOLO model: {e}")
    raise

# Data Preparation Lists
data_sequences = []
data_labels = []

def sample_frames(cap, sample_rate):
    """Samples frames from a video capture object based on the specified sample rate."""
    sampled_frames = []
    try:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            if len(sampled_frames) % sample_rate == 0:
                sampled_frames.append(frame)
    except Exception as e:
        print(f"Error sampling frames: {e}")
    
    return sampled_frames

# Helper Function for Aggregating Features
def aggregate_temporal_features(keypoints):
    """Aggregate temporal features over a sequence window for each person."""
    keypoints = np.array(keypoints)
    mean = np.mean(keypoints, axis=0)
    std = np.std(keypoints, axis=0)
    max_val = np.max(keypoints, axis=0)
    min_val = np.min(keypoints, axis=0)
    return np.concatenate([mean, std, max_val, min_val])

# Process Videos and Extract Features
for idx, video_id in enumerate(video_ids):
    label = labels[idx]
    video_path = os.path.join(video_folder, f"{video_id}.mp4")

    # Read video and sample frames
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error opening video file {video_path}")
        continue

    sampled_frames = sample_frames(cap, frame_sample_rate)
    cap.release()  # Close video file

    keypoints_sequence_person1 = []
    keypoints_sequence_person2 = []

    for frame in sampled_frames:
        # Run pose estimation on the frame
        try:
            results = pose_model(frame)
            if len(results) >= 2:
                # Each result may have keypoints for detected persons
                keypoints_person1 = results[0].keypoints.numpy() if results[0].keypoints is not None else None
                keypoints_person2 = results[1].keypoints.numpy() if results[1].keypoints is not None else None
                
                # Append keypoints for both persons if they are detected
                if keypoints_person1 is not None:
                    keypoints_sequence_person1.append(keypoints_person1)
                else:
                    print(f"No keypoints detected for Person 1 in video {video_id}.")

                if keypoints_person2 is not None:
                    keypoints_sequence_person2.append(keypoints_person2)
                else:
                    print(f"No keypoints detected for Person 2 in video {video_id}.")
        except Exception as e:
            print(f"Error processing frame in video {video_id}: {e}")

    # Aggregate Temporal Features in Windows
    num_windows = min(len(keypoints_sequence_person1), len(keypoints_sequence_person2)) // window_size
    for window_idx in range(num_windows):
        # Slice windows for each person
        window_keypoints_person1 = keypoints_sequence_person1[window_idx * window_size:(window_idx + 1) * window_size]
        window_keypoints_person2 = keypoints_sequence_person2[window_idx * window_size:(window_idx + 1) * window_size]

        # Aggregate features for each window and each person
        agg_features_person1 = aggregate_temporal_features(window_keypoints_person1)
        agg_features_person2 = aggregate_temporal_features(window_keypoints_person2)
        combined_features = np.concatenate([agg_features_person1, agg_features_person2])

        # Append to sequence data
        data_sequences.append(combined_features)
        data_labels.append(label)

# Scale and Reshape Data for Model Input
scaler = StandardScaler()
data_sequences = scaler.fit_transform(data_sequences)
data_sequences = np.array(data_sequences)
data_labels = np.array(data_labels)

# Reshape for LSTM (samples, time steps, features)
n_samples = len(data_sequences) // window_size
data_sequences = data_sequences.reshape((n_samples, window_size, -1))
data_labels = data_labels[:n_samples]

# Build LSTM Model
input_layer = Input(shape=(window_size, data_sequences.shape[2]))
lstm_out = LSTM(64, return_sequences=True)(input_layer)
lstm_out = LSTM(32)(lstm_out)
lstm_out = Dropout(0.3)(lstm_out)

# Fully Connected Layers
dense_out = Dense(64, activation='relu')(lstm_out)
dense_out = Dropout(0.2)(dense_out)
output = Dense(1, activation='sigmoid')(dense_out)  # Binary classification example

# Compile Model
model = Model(inputs=input_layer, outputs=output)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train Model
try:
    model.fit(data_sequences, data_labels, epochs=20, batch_size=8, validation_split=0.2)
except Exception as e:
    print(f"Error during model training: {e}")

# Evaluate Model
try:
    loss, accuracy = model.evaluate(data_sequences, data_labels)
    print(f"Model Accuracy: {accuracy:.2f}")
except Exception as e:
    print(f"Error during model evaluation: {e}")

OpenCV: Couldn't read video stream from file "../sdata/NFCK3A.joke1.rep1.take1.ThatsNotAHat.mp4.mp4"
OpenCV: Couldn't read video stream from file "../sdata/NFCK3A.joke1.rep1.take2.ThatsNotAHat.mp4.mp4"
OpenCV: Couldn't read video stream from file "../sdata/NFCK3A.joke1.rep2.take1.ThatsNotAHat.mp4.mp4"
OpenCV: Couldn't read video stream from file "../sdata/NFCK3A.joke1.rep3.take1.ThatsNotAHat.mp4.mp4"
OpenCV: Couldn't read video stream from file "../sdata/NFCK3A.joke2.rep1.take1.TearingPaper.mp4.mp4"
OpenCV: Couldn't read video stream from file "../sdata/NFCK3A.joke2.rep2.take1.TearingPaper.mp4.mp4"
OpenCV: Couldn't read video stream from file "../sdata/NFCK3A.joke2.rep3.take1.TearingPaper.mp4.mp4"
OpenCV: Couldn't read video stream from file "../sdata/NFCK3A.joke3.rep1.take1.NomNomNom.mp4.mp4"
OpenCV: Couldn't read video stream from file "../sdata/NFCK3A.joke3.rep2.take1.NomNomNom.mp4.mp4"
OpenCV: Couldn't read video stream from file "../sdata/NFCK3A.joke3.rep3.take1.NomNomNom.mp4.mp4"

Error opening video file ../sdata/NFCK3A.joke1.rep1.take1.ThatsNotAHat.mp4.mp4
Error opening video file ../sdata/NFCK3A.joke1.rep1.take2.ThatsNotAHat.mp4.mp4
Error opening video file ../sdata/NFCK3A.joke1.rep2.take1.ThatsNotAHat.mp4.mp4
Error opening video file ../sdata/NFCK3A.joke1.rep3.take1.ThatsNotAHat.mp4.mp4
Error opening video file ../sdata/NFCK3A.joke2.rep1.take1.TearingPaper.mp4.mp4
Error opening video file ../sdata/NFCK3A.joke2.rep2.take1.TearingPaper.mp4.mp4
Error opening video file ../sdata/NFCK3A.joke2.rep3.take1.TearingPaper.mp4.mp4
Error opening video file ../sdata/NFCK3A.joke3.rep1.take1.NomNomNom.mp4.mp4
Error opening video file ../sdata/NFCK3A.joke3.rep2.take1.NomNomNom.mp4.mp4
Error opening video file ../sdata/NFCK3A.joke3.rep3.take1.NomNomNom.mp4.mp4
Error opening video file ../sdata/NFCK3A.joke4.rep1.take1.ThatsNotACat.mp4.mp4
Error opening video file ../sdata/NFCK3A.joke4.rep2.take1.ThatsNotACat.mp4.mp4
Error opening video file ../sdata/NFCK3A.joke4.rep3.take1.Tha

OpenCV: Couldn't read video stream from file "../sdata/3ALYEd.joke2.rep3.take1.NomNomNom.mp4.mp4"
OpenCV: Couldn't read video stream from file "../sdata/3ALYEd.joke3.rep1.take1.TearingPaper.mp4.mp4"
OpenCV: Couldn't read video stream from file "../sdata/3ALYEd.joke3.rep2.take1.TearingPaper.mp4.mp4"
OpenCV: Couldn't read video stream from file "../sdata/3ALYEd.joke3.rep3.take1.TearingPaper.mp4.mp4"
OpenCV: Couldn't read video stream from file "../sdata/3ALYEd.joke4.rep1.take1.Peekaboo.mp4.mp4"
OpenCV: Couldn't read video stream from file "../sdata/3ALYEd.joke4.rep2.take1.Peekaboo.mp4.mp4"
OpenCV: Couldn't read video stream from file "../sdata/3ALYEd.joke4.rep3.take1.Peekaboo.mp4.mp4"
OpenCV: Couldn't read video stream from file "../sdata/3ALYEd.joke5.rep1.take1.ThatsNotACat.mp4.mp4"
OpenCV: Couldn't read video stream from file "../sdata/3ALYEd.joke5.rep2.take1.ThatsNotACat.mp4.mp4"
OpenCV: Couldn't read video stream from file "../sdata/3ALYEd.joke5.rep3.take1.ThatsNotACat.mp4.mp4"
OpenC

ValueError: Expected 2D array, got 1D array instead:
array=[].
Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample.